# SHRED on ME4OH full-field OFAM3 SST data

In [ ]:
from datetime import date, timedelta

import netCDF4 as nc
import pandas as pd

from Data.preprocess_sst_ofam3_shred import build_shred_datasets
from plotting_functions import plot_sst_map
import models  # https://github.com/Jan-Williams/pyshred/blob/main/models.py 

## Original data

In [ ]:
filepath = "Data/OFAM3/temp_ofam3_7d_197901-201412.0p25x0p25.nc"
file_data = nc.Dataset(filepath,'r')

In [ ]:
# Extract variables
start_time = date(1979,1,1)
dates = [start_time + timedelta(t) for t in file_data["Time"][:].data]  # Time stored as days since 1979-01-01 00:00:00
lat = file_data["yt_ocean"][:].data  # in degrees N
long = file_data["xt_ocean"][:].data  # in degrees E
depth = file_data["st_ocean"][:].data  # in metres, Z axis
temp = file_data["temp"][:].data  # in degrees C, sea_water_potential_temperature
  # temp.shape is (Time, st_ocean, yt_ocean, xt_ocean)

In [ ]:
file_data.close()

In [ ]:
temp.shape

In [ ]:
for k in range(len(dates)):
    print(f"Running for timestamp {dates[k]}...")
    sst_array = []
    lat_array = []
    long_array = []
    date_array = []
    for i in range(len(lat)):
        for j in range(len(long)):
            sst_array.append(temp[k][0][i][j])
            lat_array.append(lat[i])
            long_array.append(long[j])
            date_array.append(dates[k])
    df = pd.DataFrame({
        "Date": date_array,
        "Latitude": lat_array,
        "Longitude": long_array,
        "SST": sst_array
    })

    # Convert longitude to -180 to 180 format, allows plotting against geopandas world maps
    print("Converting longitudes...")
    df['Longitude'] = df['Longitude'].apply(lambda x: -(360-x) if x > 180 else x)
    # Remove land values
    land_indexes = df.index[df['SST'] == -32768].tolist()
    df = df.loc[~df.index.isin(land_indexes)]

    # Save world file
    # df.to_csv(f"Data/OFAM3/world/{dates[k]}.csv", index=False)

    # Filter to NA basin
    print("Filtering NA basin...")
    na_df = df.loc[(df['Longitude'] >= -80) & (df['Longitude'] <= 0)
                        & (df['Latitude'] <= 60) & (df['Latitude'] >= 0)
                        ].sort_values('Date').reset_index(drop=True)
    na_file_name = f"Data/OFAM3/NA/{dates[k]}.csv"
    # na_df.to_csv(na_file_name, index=False)

    

    break

In [ ]:
df

In [ ]:
600*1400

In [ ]:
depth

In [ ]:
temp[0][0]

In [ ]:
long.shape

In [ ]:
lat.shape

In [ ]:
DATA_PATH = 'Data/ME4OH_EN411_OFAM3/data/en4.1.1/1993-2014/ofam3-jra55.all.EN.4.1.1.f.profiles.g10.199301.update.extra.anom.199301_201412.nc'
load_and_filter_ohc(DATA_PATH)

# Example visualisation

In [ ]:
date = '2012-06-28'  
date = '1979-01-04'
df = pd.read_csv(f"Data/OFAM3/NA/{date}.csv")
# df['Date'] = pd.to_datetime('1979-01-04')

In [ ]:
df

In [ ]:
plot_sst_map(df, date, min(df.SST), max(df.SST), basin='NA')

# SHRED

In [ ]:
num_sensors = 3

In [ ]:
train, val, test, dates, lats, longs = build_shred_datasets(num_sensors=num_sensors)

In [ ]:
shred = models.SHRED(num_sensors, len(lats), hidden_size=64, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
validation_errors = models.fit(shred, train_dataset, valid_dataset, batch_size=64, num_epochs=1000, lr=1e-3, verbose=True, patience=5)

In [ ]:
test_recons = sc.inverse_transform(shred(test_dataset.X).detach().cpu().numpy())
test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())
print('Test Reconstruction Error: ')
print(np.linalg.norm(test_recons - test_ground_truth) / np.linalg.norm(test_ground_truth))

In [ ]:
test_recons